# Ollama Generation Benchmark

Compare the generation sweep while holding the embedding model fixed.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

from eval.model_benchmark import ModelBenchmarkConfig, run_model_benchmark
from helpers.experiment_models import DEFAULT_EMBEDDING_MODEL, GENERATION_MODEL_SWEEP

os.environ.setdefault("OLLAMA_ENDPOINT", "http://127.0.0.1:11434")
os.environ.setdefault("INPUT_BASE_DIR", str(repo_root / "data" / "evidence" / "mimic_discharge_subset"))

generation_model_sweep = list(GENERATION_MODEL_SWEEP)
embedding_model = os.environ.get("BENCHMARK_EMBEDDING_MODEL", DEFAULT_EMBEDDING_MODEL)
sample_size = 5
use_umls = os.environ.get("UMLS_ENABLED", "true").strip().lower() == "true"
schema_guided = os.environ.get("INDEX_SCHEMA_GUIDED", "false").strip().lower() == "true"
mimic_csv = repo_root / "data" / "mimic_iv_note" / "discharge.csv"

print("OLLAMA_ENDPOINT:", os.environ["OLLAMA_ENDPOINT"])
print("generation_model_sweep:", generation_model_sweep)
print("embedding_model:", embedding_model)


In [ ]:
results = await run_model_benchmark(
    ModelBenchmarkConfig(
        input_dir=Path(os.environ["INPUT_BASE_DIR"]),
        output_root=repo_root / "output" / "ollama_model_benchmark",
        generation_models=tuple(generation_model_sweep),
        embedding_model=embedding_model,
        use_umls=use_umls,
        schema_guided=schema_guided,
        mimic_csv=(mimic_csv if mimic_csv.exists() else None),
        sample_size=sample_size,
    )
)

pd.DataFrame(result.__dict__ for result in results).sort_values(["exact_match", "mean_query_seconds"], ascending=[False, True])
